In [1]:
import pandas as pd

# Cargar el archivo CSV
df = pd.read_csv("../data/raw/ruido.csv", delimiter=";", on_bad_lines='warn')


In [2]:
df.head()

print(df.tail().to_latex(index=False,
                  formatters={"name": str.upper},
                  float_format="{:.1f}".format,
)) 

\begin{tabular}{rrrrlllllll}
\toprule
NMT & Año & mes & dia & tipo & LAeq & L1 & L10 & L50 & L90 & L99 \\
\midrule
55 & 2025 & 12 & 1 & T & 69,9 & 83,5 & 71,3 & 59,7 & 46 & 39,4 \\
86 & 2025 & 12 & 1 & D & 56,4 & 65,4 & 59,6 & 52,1 & 45,8 & 42 \\
86 & 2025 & 12 & 1 & E & 53,3 & 63,3 & 56,5 & 49,4 & 44,4 & 41,2 \\
86 & 2025 & 12 & 1 & N & 45,9 & 57,8 & 47 & 38,7 & 36 & 34,5 \\
86 & 2025 & 12 & 1 & T & 54,2 & 64,2 & 57,8 & 48,5 & 37,4 & 35 \\
\bottomrule
\end{tabular}



In [3]:
# Mantener solo las observaciones de Plaza Elíptica
CODIGO_PLAZA_ELIPTICA = 14

df = df[df['NMT'].isin([CODIGO_PLAZA_ELIPTICA])]

In [4]:
# Construir columna Fecha
# TODO: Revisar warning
df["FECHA"] = pd.to_datetime(
    {
        "year": df["Año"],
        "month": df["mes"],
        "day": df["dia"]
    },
    errors="coerce"   # <-- converts invalid dates into NaT instead of raising an error
)

In [5]:
# Reordenar las columnas
cols = ['FECHA'] + [col for col in df.columns if col != 'FECHA']
df = df.reindex(columns=cols)

In [6]:
# Eliminar columnas innecesarias
df.drop(columns=[
   'Año', 
   'mes', 
   'dia', 
], inplace=True)


In [ ]:
# Pivot de tipo
df = (
    df
    .pivot(index=["FECHA", "NMT"], columns="tipo", values=["LAeq", "L1", "L10", "L50", "L90", "L99"])
    .reset_index()
)

df = df.rename(columns={
    "D": "Diurno",
    "E": "Vespertino",
    "N": "Nocturno",
    "T": "Total"
})

df = df.sort_values(["FECHA"]).reset_index(drop=True)

In [ ]:
# Eliminar multi-index en las columnas

new_cols = []

for col in df.columns:
    if isinstance(col, tuple):
        main, sub = col
        new_cols.append(f"{main}{sub}")  # e.g., "L90" + "Diurno" → "L90Diurno"
    else:
        new_cols.append(col)

df.columns = new_cols


In [9]:
# Eliminar comas y convertir a float 

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(",", ".").astype(float)


In [ ]:
# Crear el perfil del dataset
# from ydata_profiling import ProfileReport

# profile = ProfileReport(df, title="AEDA_Ruido_OK", explorative=True)
# profile.to_file("AEDA_Ruido_OK.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:00<00:00, 219.25it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# NO₂
plt.plot(df["FECHA"], df["L99Total"], label="L99 Total")

# PM₂.₅
plt.plot(df["FECHA"], df["L1Total"], label="L1 Total")

plt.xlabel("Fecha")
plt.ylabel("Concentración")
plt.title("Evolución diaria de L99 Total y L1 Total")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("l99total_l1total_timeseries.png", dpi=300)


In [12]:
print(df.head().to_latex(index=False,
                  formatters={"name": str.upper},
                  float_format="{:.1f}".format,
))  

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
FECHA & NMT & LAeqDiario & LAeqVespertino & LAeqNocturno & LAeqTotal & L1Diario & L1Vespertino & L1Nocturno & L1Total & L10Diario & L10Vespertino & L10Nocturno & L10Total & L50Diario & L50Vespertino & L50Nocturno & L50Total & L90Diario & L90Vespertino & L90Nocturno & L90Total & L99Diario & L99Vespertino & L99Nocturno & L99Total \\
\midrule
2014-01-01 00:00:00 & 14 & 65.1 & 66.3 & 68.5 & 66.7 & 70.5 & 70.8 & 75.1 & 71.6 & 67.6 & 68.2 & 67.5 & 67.7 & 63.3 & 65.0 & 62.6 & 63.5 & 58.2 & 62.1 & 57.4 & 58.5 & 55.3 & 59.5 & 53.8 & 55.3 \\
2014-01-02 00:00:00 & 14 & 68.7 & 67.5 & 61.2 & 67.1 & 76.1 & 72.1 & 69.2 & 74.2 & 70.5 & 69.4 & 64.9 & 69.8 & 67.5 & 66.3 & 58.9 & 65.8 & 64.6 & 63.3 & 52.1 & 56.1 & 63.0 & 61.2 & 47.3 & 49.2 \\
2014-01-03 00:00:00 & 14 & 67.5 & 68.9 & 61.5 & 66.7 & 72.2 & 73.6 & 68.8 & 72.0 & 69.6 & 70.4 & 65.0 & 69.4 & 66.5 & 67.3 & 59.6 & 65.4 & 63.5 & 64.3 & 54.1 & 57.3 & 62.0 & 62.4 & 50.7 & 52.1 \\
2014-01-04 00:00: